# 4.7 · 广义线性模型 / Generalized Linear Models (GLM)

> **课程定位 / Where this fits**
> 第 7 课，**Part 4 · 监督学习：回归**。
> Lesson 7, **Part 4 · Supervised Regression**.
>
> 前面的线性回归假设目标是**连续值 + 正态误差**。但现实里目标常常不是：理赔**次数**(非负整数)、点击**与否**(0/1)、保险**金额**(正且右偏)。**GLM** 用一个统一框架把线性回归推广到这些情形——它也是理解**逻辑回归(5.1)** 的关键(逻辑回归就是 GLM 的一个特例)。
> Linear regression assumed a **continuous target with normal errors**. Reality often differs: claim **counts** (non-negative integers), click **or not** (0/1), claim **amounts** (positive, right-skewed). **GLM** generalizes linear regression to all of these in one framework — and it's the key to understanding **logistic regression (5.1)**, which is just a special case of GLM.
>
> 💼 **实战/面试视角**："计数数据怎么建模 / 为什么不用线性回归 / GLM 是什么" 偏统计/精算/风控岗。
> 💼 **Practical/interview angle:** "modeling count data / why not linear regression / what is a GLM" — stats/actuarial/risk roles.

> 📐 **符号约定 / Notation**
> - 连接函数 link $g$: $g(\mathbb E[y]) = \mathbf{x}^\top\mathbf{w}$ —— 把均值映到线性预测器
> - 分布族 family —— 高斯/泊松/伽马/二项 / the exponential-family distribution

> 💡 **面试相关 / Interview-relevant**
> - "GLM 的三要素（分布族 + 连接函数 + 线性预测器）"（出镜率 ★★★★）
> - "计数数据为什么用泊松回归"（★★★★★）
> - "log 连接下系数怎么解读（乘性效应）"（★★★★）
> - "逻辑回归/线性回归都是 GLM 的特例"（★★★★★）

---

## 学习目标 / Learning Objectives

1. 理解为什么线性回归不适合计数/比例/偏态目标。
   Understand why linear regression fails on count/proportion/skewed targets.
2. 掌握 GLM 三要素：分布族 + 连接函数 + 线性预测器。
   Master GLM's three parts: family + link + linear predictor.
3. 用**泊松回归**建模计数数据，理解 log 连接的乘性解读。
   Model counts with **Poisson regression** and read log-link coefficients multiplicatively.
4. 用**伽马回归**建模正偏态金额。
   Model positive skewed amounts with **Gamma regression**.
5. 看清**线性回归/逻辑回归都是 GLM 特例**。
   See that linear and logistic regression are GLM special cases.

## 目录 / TOC
1. [先建直觉 + GLM 三要素 ⭐](#1)
2. [📋 计数数据：OLS 的失败](#2)
3. [泊松回归 ⭐](#3)
4. [伽马回归：偏态金额 ⭐](#4)
5. [统一视角：都是 GLM ⭐](#5)
6. [小结](#6)


<a id="1"></a>
## 1. 先建直觉 + GLM 三要素 ⭐ / Intuition & the Three Parts

线性回归 $\hat y = \mathbf{x}^\top\mathbf{w}$ 直接预测目标，问题是：它的输出可以是任意实数（包括负数），且假设误差正态、方差恒定。**计数不能为负、概率必须在 [0,1]、金额右偏且方差随均值增大**——硬套线性回归就会出现"预测负理赔次数"之类的荒谬结果。
Linear regression $\hat y = \mathbf{x}^\top\mathbf{w}$ predicts the target directly, but its output can be any real number (including negatives), assuming normal errors with constant variance. **Counts can't be negative, probabilities must be in [0,1], amounts are skewed with variance growing with the mean** — forcing linear regression gives absurd results like "negative predicted claim counts".

**GLM 的三要素**（面试要点）：
**GLM's three parts** (interview point):
1. **分布族(family)**：目标服从什么分布——高斯(连续)、泊松(计数)、伽马(正偏态)、二项(0/1)。
   **Family:** the target's distribution — Gaussian (continuous), Poisson (counts), Gamma (positive skewed), Binomial (0/1).
2. **连接函数(link) $g$**：把目标的**均值**映到线性预测器：$g(\mathbb E[y]) = \mathbf{x}^\top\mathbf{w}$。如 log 连接保证均值恒为正，logit 连接保证概率在 [0,1]。
   **Link $g$:** maps the target's **mean** to the linear predictor: $g(\mathbb E[y]) = \mathbf{x}^\top\mathbf{w}$. log-link keeps the mean positive; logit-link keeps probabilities in [0,1].
3. **线性预测器**：$\mathbf{x}^\top\mathbf{w}$——还是熟悉的线性组合。
   **Linear predictor:** $\mathbf{x}^\top\mathbf{w}$ — the familiar linear combination.

换不同的(分布, 连接)，就得到不同的模型，但都用同一套算法(IRLS)拟合。
Different (family, link) choices give different models, all fit by the same algorithm (IRLS).


<a id="2"></a>
## 2. 计数数据：OLS 的失败 / Count Data: OLS Fails

用一个保险场景：理赔**次数**随风险评分增加。先看直接用 OLS 会出什么问题。
An insurance scenario: claim **counts** rise with a risk score. First, what goes wrong with plain OLS.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
sns.set_theme(style="whitegrid")
rng = np.random.default_rng(42)

# 造计数数据: 真实 log(理赔率) 线性于 risk, 理赔次数服从泊松 / Poisson counts
n = 1000
risk = rng.uniform(0, 3, n)
true_rate = np.exp(-0.5 + 0.8*risk)        # 真实率(恒正, 因为是 exp)
claims = rng.poisson(true_rate)             # 按泊松分布采样计数

ols = sm.OLS(claims, sm.add_constant(risk)).fit()   # 直接用 OLS 拟合
x_plot = np.linspace(0, 3, 100)
ols_pred = ols.predict(sm.add_constant(x_plot))

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(risk, claims, alpha=0.2, s=10, label="理赔次数(计数) counts")
ax.plot(x_plot, ols_pred, "r-", lw=2, label="OLS 拟合")
ax.axhline(0, color="k", lw=0.5); ax.legend()
ax.set_xlabel("risk score"); ax.set_ylabel("claims")
ax.set_title("OLS 对计数数据: 低 risk 处预测负理赔次数(无意义!)")
plt.tight_layout(); plt.show()
print(f"OLS 在 risk=0 处预测理赔次数 = {ols_pred[0]:.2f}  ← 负数! 计数不可能为负")
print("且计数数据方差随均值增大(泊松性质) → OLS 的同方差假设也违反")


<a id="3"></a>
## 3. 泊松回归 ⭐ / Poisson Regression

计数数据的标准 GLM：**分布族=泊松，连接函数=log**。log 连接意味着 $\log(\mathbb E[y]) = \mathbf{x}^\top\mathbf{w}$，即 $\mathbb E[y] = e^{\mathbf{x}^\top\mathbf{w}}$——**预测值永远为正**，且自然捕捉"指数增长"。
The standard GLM for counts: **family = Poisson, link = log**. The log-link means $\log(\mathbb E[y]) = \mathbf{x}^\top\mathbf{w}$, i.e. $\mathbb E[y] = e^{\mathbf{x}^\top\mathbf{w}}$ — **predictions are always positive** and naturally capture exponential growth.

**log 连接下系数的解读是"乘性"的**(重要)：特征每增加 1 单位，目标均值**乘以** $e^{w}$ 倍（而非加上 $w$）。
**Under the log-link, coefficients are read multiplicatively** (important): a one-unit increase in a feature **multiplies** the mean by $e^{w}$ (rather than adding $w$).


In [ ]:
# 泊松 GLM: family=Poisson(默认 log 连接) / Poisson GLM with log link
poisson_glm = sm.GLM(claims, sm.add_constant(risk), family=sm.families.Poisson()).fit()
poisson_pred = poisson_glm.predict(sm.add_constant(x_plot))

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(risk, claims, alpha=0.2, s=10, label="理赔次数")
ax.plot(x_plot, ols.predict(sm.add_constant(x_plot)), "r--", lw=1.5, label="OLS(会预测负数)")
ax.plot(x_plot, poisson_pred, "g-", lw=2.5, label="泊松回归(永远>0)")
ax.plot(x_plot, np.exp(-0.5+0.8*x_plot), "k:", lw=1.5, label="真实率 true")
ax.axhline(0, color="k", lw=0.5); ax.legend()
ax.set_xlabel("risk"); ax.set_ylabel("claims")
ax.set_title("泊松回归: 指数曲线, 永远为正, 贴合真实")
plt.tight_layout(); plt.show()
print(f"泊松系数: 截距={poisson_glm.params[0]:.3f}(真-0.5), risk={poisson_glm.params[1]:.3f}(真0.8) → 准确恢复!")
print(f"💡 log 连接乘性解读: risk 每+1, 理赔率 ×e^{poisson_glm.params[1]:.2f}={np.exp(poisson_glm.params[1]):.2f} 倍")


<a id="4"></a>
## 4. 伽马回归：偏态金额 ⭐ / Gamma Regression

理赔**金额**：恒为正、严重右偏、方差随均值增大。这类数据用 **分布族=伽马，连接=log** 的 GLM。OLS 会被极端大额拉偏、甚至预测负金额。
Claim **amounts**: positive, heavily right-skewed, variance growing with the mean. Use a GLM with **family = Gamma, link = log**. OLS is dragged by extreme large amounts and can even predict negatives.

> 保险定价的标准做法：**频率(次数)用泊松、金额用伽马**，两者相乘得到期望损失（合起来即 **Tweedie** 模型）。
> Standard insurance pricing: **frequency (counts) via Poisson, severity (amount) via Gamma**; their product is the expected loss (jointly the **Tweedie** model).


In [ ]:
# 造右偏正值金额: log(均值) 线性于 risk, 金额服从伽马 / right-skewed positive amounts
true_mean_amount = np.exp(6 + 0.5*risk)
amount = rng.gamma(shape=2.0, scale=true_mean_amount/2.0)   # 伽马: 均值 = shape×scale
print(f"理赔金额: 全部>0, 右偏(偏度 skew={pd.Series(amount).skew():.1f})")

# 伽马 GLM(log 连接) vs OLS / Gamma GLM vs OLS
gamma_glm = sm.GLM(amount, sm.add_constant(risk),
                   family=sm.families.Gamma(link=sm.families.links.Log())).fit()
ols_amt = sm.OLS(amount, sm.add_constant(risk)).fit()

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(risk, amount, alpha=0.15, s=10, label="理赔金额")
ax.plot(x_plot, ols_amt.predict(sm.add_constant(x_plot)), "r--", lw=1.5, label="OLS")
ax.plot(x_plot, gamma_glm.predict(sm.add_constant(x_plot)), "g-", lw=2.5, label="伽马回归 Gamma")
ax.plot(x_plot, np.exp(6 + 0.5*x_plot), "k:", lw=1.5, label="真实均值 true")
ax.legend(); ax.set_xlabel("risk"); ax.set_ylabel("claim amount")
ax.set_title("伽马回归: 捕捉指数增长 + 正偏态")
plt.tight_layout(); plt.show()
print("伽马回归对右偏正值金额更合适; OLS 被极端大额拉偏 + 可能预测负金额")
print("→ 保险定价标准: 频率用泊松, 金额用伽马(合起来是 Tweedie 模型)")


<a id="5"></a>
## 5. 统一视角：都是 GLM ⭐ / The Unifying View

GLM 的真正威力是**统一**：换一组(分布, 连接)就得到不同模型，全用同一套算法拟合。
GLM's real power is **unification**: swap the (family, link) pair to get different models, all fit by the same algorithm.

| 模型 | 分布族 family | 连接 link | 用于 |
|---|---|---|---|
| **线性回归** | 高斯 Gaussian | identity | 连续目标 |
| **逻辑回归(5.1)** | 二项 Binomial | logit | 0/1 分类 |
| **泊松回归** | Poisson | log | 计数 |
| **伽马回归** | Gamma | log | 正偏态金额 |

下面验证：**GLM(高斯, identity) 就是普通线性回归**，系数完全一致。
Below we verify: **GLM(Gaussian, identity) is exactly ordinary linear regression**, with identical coefficients.


In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import LinearRegression

d = fetch_california_housing(as_frame=True)
Xh, yh = d.data.values[:2000], d.target.values[:2000]
# GLM(高斯+identity) 应等于 LinearRegression / should be identical
glm_gaussian = sm.GLM(yh, sm.add_constant(Xh), family=sm.families.Gaussian()).fit()
lr = LinearRegression().fit(Xh, yh)
print("GLM(高斯, identity) 系数 vs LinearRegression 系数:")
print(f"最大差异 = {np.abs(glm_gaussian.params[1:] - lr.coef_).max():.8f}")
print("→ 完全相同! 普通线性回归 = GLM 的(高斯, identity link)特例")
print("GLM 把一切统一: 改(分布,link)就得到不同模型, 同一套 IRLS 算法拟合")


<a id="6"></a>
## 6. 小结 / Summary

```
GLM 三要素: 分布族(family) + 连接函数(link, g(E[y])=xᵀw) + 线性预测器(xᵀw)
线性回归假设连续+正态, 不适合: 计数(可负)/概率(超[0,1])/偏态金额(方差变)
泊松回归: family=Poisson, log 连接 → 预测恒正; 系数乘性解读(每+1单位 ×e^w)
伽马回归: family=Gamma, log 连接 → 正偏态金额(保险: 频率泊松+金额伽马=Tweedie)
统一: 线性回归=(高斯,identity), 逻辑回归=(二项,logit), 都是 GLM 特例, 同一套 IRLS
```

### 💡 面试速查 / Interview cheat-sheet
1. **GLM = 分布族 + 连接函数 + 线性预测器**。
   GLM = family + link + linear predictor.
2. **计数用泊松回归**(log 连接保证恒正); 系数是**乘性**效应。
   Counts → Poisson regression (log-link keeps positive); coefficients are multiplicative.
3. **偏态正值(金额)用伽马回归**。
   Positive skewed amounts → Gamma regression.
4. **线性回归=(高斯,identity), 逻辑回归=(二项,logit)** 都是 GLM 特例。
   Linear = (Gaussian, identity), logistic = (Binomial, logit), both GLM cases.
5. log 连接下"特征+1 → 均值×e^w"(乘性), 不是加性。
   Under log-link, "+1 feature → ×e^w mean" (multiplicative, not additive).

### 下一节 / Next
**4.8 非线性回归**——GLM 仍是"线性预测器"。如果目标和特征是真正的非线性参数关系(如指数衰减、S 曲线), 就要用非线性最小二乘直接拟合曲线参数。
**4.8 Nonlinear Regression** — GLM still has a linear predictor. For genuinely nonlinear parametric relationships (exponential decay, S-curves), fit the curve parameters directly via nonlinear least squares.
